# Alpha Research — Hyperliquid seconds features

Goal: empirically test whether candidate signals from
`data/seconds_feature_engine.py` have predictive power on forward
returns after costs.

**No live trading is launched from this notebook.** It only reads
`logs/seconds_features.csv` produced by the engine when running with
`config/presets/paper_500_alpha_research.json`.

See `docs/ALPHA_MODELS_THEORY.md` for the underlying maths and
`docs/ALPHA_RESEARCH_FRAMEWORK.md` for the full methodology.

In [ ]:
from pathlib import Path
import sys
_ROOT = Path('.').resolve()
if (_ROOT / 'research').exists():
    REPO_ROOT = _ROOT
else:
    REPO_ROOT = _ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from research import alpha_metrics as am

FEATURES_PATH = REPO_ROOT / 'logs' / 'seconds_features.csv'
REPORT_PATH = REPO_ROOT / 'reports' / 'alpha_research_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Features path :', FEATURES_PATH)
print('Exists        :', FEATURES_PATH.exists())

## 1. Load + forward returns

If `logs/seconds_features.csv` is missing, first run :

```
python engine_v9.py --paper --config config/presets/paper_500_alpha_research.json
```

In [ ]:
if not FEATURES_PATH.exists():
    print('No features CSV yet. Run the engine first; see top-of-notebook command.')
    df = pd.DataFrame()
else:
    df = am.load_features(str(FEATURES_PATH))
    df = am.add_forward_returns(df)
    df = am.add_btc_residual(df)
    print('rows:', len(df), 'symbols:', sorted(df['symbol'].unique()))
df.head()

## 2. Candidate signals — sanity

In [ ]:
if not df.empty:
    sigs = [s for s in am.DEFAULT_CANDIDATE_SIGNALS if s in df.columns]
    print('Available signals:', sigs)
    df[sigs].describe().T if sigs else None

## 3. Information Coefficient (raw forward returns)

In [ ]:
ic_raw = am.compute_ic_table(df) if not df.empty else pd.DataFrame()
ic_raw

## 4. IC on BTC-residualized forward returns

If a signal only predicts the raw fwd return but loses IC on the
residual, it's a **beta proxy**, not an alpha.

In [ ]:
ic_resid = am.compute_ic_table(df, residual=True) if not df.empty else pd.DataFrame()
ic_resid

## 5. Bucket analysis

Decile buckets of the signal vs mean forward return + hit rate.

In [ ]:
best = None
if not df.empty and not ic_raw.empty:
    ic_pick = ic_raw.dropna(subset=['ic_spearman']).copy()
    if not ic_pick.empty:
        ic_pick['abs_ic'] = ic_pick['ic_spearman'].abs()
        best = ic_pick.sort_values('abs_ic', ascending=False).iloc[0]
        print('Best raw IC :', best['signal'], 'h=', int(best['horizon_s']),
              'spearman=', best['ic_spearman'])
if best is not None:
    am.bucket_analysis(df, best['signal'], int(best['horizon_s']))

## 6. Threshold simulation (long top decile / short bottom decile)

Realistic costs : 8, 12, 16 bps round-trip — typical paper executor
fees + slippage on Hyperliquid.

In [ ]:
if not df.empty:
    scan = am.scan_signals(df)
    scan = scan.sort_values('net_bps', ascending=False).reset_index(drop=True)
    scan.head(30)

## 7. Walk-forward validation

Train window = 3600 s, test window = 600 s, rolling. Mean and median
net bps per fold ; fraction of positive folds.

In [ ]:
if not df.empty and best is not None:
    wf = am.walk_forward(df, best['signal'], int(best['horizon_s']))
    wf

## 8. Uniqueness test

Residualize each signal against (spread_bps, rv_30s, obi_5,
trade_imbalance_10s). If IC of the residualized signal vs forward
return stays non-zero, the signal carries information beyond those
controls.

In [ ]:
uniq_rows = []
if not df.empty:
    for sig in am.DEFAULT_CANDIDATE_SIGNALS:
        if sig in df.columns:
            uniq_rows.append(am.signal_residual_ic(df, sig, 60))
pd.DataFrame(uniq_rows)

## 9. Export Markdown report

Same content as `python scripts/run_alpha_research.py`.

In [ ]:
import subprocess, shlex
cmd = f'python {shlex.quote(str(REPO_ROOT / "scripts" / "run_alpha_research.py"))} --features {shlex.quote(str(FEATURES_PATH))} --out {shlex.quote(str(REPORT_PATH))}'
print(cmd)
if FEATURES_PATH.exists():
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
else:
    print('Skipped — features CSV not present.')

## 10. Validation criteria — reminder

A signal becomes a candidate alpha **only when** :
1. Spearman IC stable and non-zero across days.
2. Bucket analysis monotone (or at least coherent and not driven
   by a single bucket).
3. Threshold-sim net bps positive after costs.
4. Walk-forward profit factor > 1, folds mostly positive.
5. BTC-residual IC non-zero (not just beta).
6. Residual IC vs spread / vol / OBI / TI controls non-zero.
7. Performance not concentrated on a single day or single coin.

**Until all these gates are met, leave the related strategy
`enabled=false` in the config.**